# Component 3 - XAI step 2: can each biomarker be trusted, and why does it have this value?

* **Reliability** of every measured cell: GREEN / YELLOW (low StarDist confidence) / RED (implausible size or shape).
  Tested on the 40 annotated crops, each predicted by the Phase 2 fold model that never saw it:
  does low confidence find the cells that are really wrong? (primary: AUC, 95 % CI above 0.5)
* **Evidence**: cells coloured by area (why this CV) and by side count (why this HEX).
* **Uncertainty**: 95 % CI of ECD / CV / HEX (bootstrap over 64 px blocks of cells), compared with Konan's
  ~110-cell sampling uncertainty.

Rules fixed in `config.yaml -> xai.biomarkers` before any result. Nothing here changes a biomarker number.

**GPU runtime** (Runtime > Change runtime type > T4 GPU) for cells 3 and 5; cell 4 is CPU.

In [ ]:
# 1. Setup (mount Drive, pull latest code, copy images, set paths)
from google.colab import drive; drive.mount("/content/drive")
REPO_DIR = "/content/drive/MyDrive/Colab Notebooks/Research_Project/xai-cornea-grading"
!git -C "{REPO_DIR}" checkout -- ml/cellular/notebooks/
!git -C "{REPO_DIR}" pull --ff-only
%run "{REPO_DIR}/ml/cellular/colab_setup.py"

In [ ]:
# 2. Install StarDist (not in Colab by default) + unit tests
!pip -q install tensorflow tf-keras stardist
!python -m pytest tests/test_xai_biomarkers.py -q

In [ ]:
# 3. Reliability vs real segmentation errors on the 40 crops (GPU, ~5 min: 5 fold models)
!python -m xai.biomarker_explain --crops 2>&1 | grep -v -i warning

In [ ]:
# 4. 95 % CIs for all validation images vs Konan's sampling uncertainty (CPU, ~1 min)
!python -m xai.biomarker_explain --intervals

In [ ]:
# 5. Explanation figures for 6 validation images spread over ECD (GPU)
!python -m xai.biomarker_explain --figures 2>&1 | grep -v -i warning

In [ ]:
# 6. Show the figures
import glob
from IPython.display import Image, display
for f in sorted(glob.glob(f"{OUTPUT_ROOT}/xai/biomarkers/figures/*.png")):
    display(Image(f, width=1500))